In [1]:
import pandas as pd
import numpy as np


In [2]:
master_file = "D:/Tushar/series_part_mapping.xlsx"
# prod_file   = "production data.xlsx"
tool_file   = "D:/Tushar/Tool list of Mould fixation-VT.xlsx"

df_master    = pd.read_excel(master_file, sheet_name="Master Sheet")
df_inventory = pd.read_excel(master_file, sheet_name="Inventory ")
df_sheet10   = pd.read_excel(master_file, sheet_name="Sheet10")
df_fg        = pd.read_excel(master_file, sheet_name="Sheet4")
df_cycle     = pd.read_excel(master_file, sheet_name="Sheet3")
# df_prod = pd.read_excel(prod_file)
df_tool = pd.read_excel(tool_file)


In [3]:
print(df_master.columns)
print(df_cycle.columns)


Index(['Series', 'Part', 'count', 'Mounlding Parts', 'FG Quantity',
       'Monthely Requirement-jan', 'Minimum Requirement', 'Dispatch'],
      dtype='object')
Index([    'Work Center Name',             'Material', 'Material Description',
                      'Plant',        'Operation NO.',        'Group Counter',
                'Control Key', 'Operation short Text',              'Machine',
                     'Labour',            'Duplicate',    2025-01-01 00:00:00,
                'Unnamed: 12'],
      dtype='object')


In [4]:
def normalize_columns(df):
    df = df.copy()
    df.columns = (
        df.columns
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace(".", "", regex=False)
    )
    return df

def norm(x):
    if pd.isna(x):
        return ""
    return str(x).strip().lower()


In [5]:
df_master    = normalize_columns(df_master)
df_inventory = normalize_columns(df_inventory)
df_sheet10   = normalize_columns(df_sheet10)
df_fg        = normalize_columns(df_fg)
# df_prod      = normalize_columns(df_prod)
df_tool      = normalize_columns(df_tool)
df_cycle     = normalize_columns(df_cycle)

In [6]:
def get_fg_stock_by_series(series):
    s = norm(series)

    r = df_inventory[df_inventory["material"].astype(str).apply(norm) == s]
    if not r.empty and not pd.isna(r.iloc[0]["unrestricted"]):
        return r.iloc[0]["unrestricted"]

    r = df_sheet10[df_sheet10["material"].astype(str).apply(norm) == s]
    if not r.empty and not pd.isna(r.iloc[0]["unrestricted"]):
        return r.iloc[0]["unrestricted"]

    r = df_fg[df_fg["series"].astype(str).apply(norm) == s]
    if not r.empty and not pd.isna(r.iloc[0]["fg_quantity"]):
        return r.iloc[0]["fg_quantity"]

    return 0


In [7]:
def get_machine_by_series(series):
    s = norm(series)
    r = df_tool[df_tool["part_no"].astype(str).apply(norm) == s]
    if not r.empty:
        return r.iloc[0]["machine_no"]
    return None


In [8]:
def get_cycle_time_by_series(series):
    s = norm(series)
    r = df_cycle[df_cycle["material"].astype(str).apply(norm) == s]

    if r.empty:
        return None

    cycle_time = r.iloc[0]["machine"]

    if pd.isna(cycle_time) or cycle_time <= 0:
        return None

    return cycle_time   # hours per unit

In [9]:
def get_time_required_by_series(series, qty):
    cycle_time = get_cycle_time_by_series(series)
    if cycle_time is None:
        return None

    return qty * cycle_time   # hours

In [10]:
MAX_MINUTES = 22 * 60
machine_load = {}
PLAN = []


In [11]:
rejection_log = []
PLAN = []
machine_load = {}
MAX_MINUTES = 22 * 60  # 22 hours

for i, row in df_master.iterrows():

    if i % 5000 == 0:
        print(f"Processed {i} rows...")

    series = row["series"]

    monthly_req = row["monthely_requirement-jan"] if not pd.isna(row["monthely_requirement-jan"]) else 0
    min_req     = row["minimum_requirement"] if not pd.isna(row["minimum_requirement"]) else 0
    dispatch    = row["dispatch"] if not pd.isna(row["dispatch"]) else 0

    fg_stock = get_fg_stock_by_series(series)
    available_fg = fg_stock - dispatch

    # BUFFER LOGIC (FINAL)
    if available_fg >= min_req:
        rejection_log.append((series, "FG >= buffer"))
        continue

    planned_qty = min_req - available_fg

    if planned_qty <= 0:
        rejection_log.append((series, "Planned qty <= 0"))
        continue

    daily_demand = monthly_req / 28 if monthly_req > 0 else 0

# OPTIONAL batching cap (does NOT use net_req)
    if daily_demand > 0 and daily_demand <= 50:
        planned_qty = min(planned_qty, 5 * daily_demand)



    if planned_qty <= 0:
        rejection_log.append((series, "Planned qty <= 0"))
        continue

    machine = get_machine_by_series(series)
    if machine is None:
        rejection_log.append((series, "Machine not found"))
        continue

    cycle_time = get_cycle_time_by_series(series)
    if cycle_time is None:
        rejection_log.append((series, "Cycle time missing"))
        continue

    time_mins = planned_qty * cycle_time * 60
    used_mins = machine_load.get(machine, 0)

    if used_mins + time_mins > MAX_MINUTES:
        rejection_log.append((series, "Machine capacity exceeded"))
        continue

    machine_load[machine] = used_mins + time_mins

    PLAN.append({
        "Machine": machine,
        "Series": series,
        "Qty": int(round(planned_qty)),
        "Time Required (hrs)": round(time_mins / 60, 2)
    })

Processed 0 rows...
Processed 5000 rows...
Processed 10000 rows...
Processed 15000 rows...
Processed 20000 rows...
Processed 25000 rows...
Processed 30000 rows...
Processed 35000 rows...
Processed 40000 rows...
Processed 45000 rows...
Processed 50000 rows...
Processed 55000 rows...
Processed 60000 rows...
Processed 65000 rows...
Processed 70000 rows...
Processed 75000 rows...
Processed 80000 rows...
Processed 85000 rows...
Processed 90000 rows...
Processed 95000 rows...
Processed 100000 rows...
Processed 105000 rows...
Processed 110000 rows...


In [12]:
df_plan = pd.DataFrame(PLAN)
df_plan = df_plan.sort_values("Machine").reset_index(drop=True)
df_plan


,Machine,Series,Qty,Time Required (hrs)
0,M.P-09,S11332-002A0X,0,10.00
1,M.P-09,S11332-002A0X,0,10.00
2,M.P-16,S31605-002A0X,0,14.29
3,M.P-16,S31851-002A0X,0,2.57
4,M.P-16,S31851-002A0X,0,2.57


In [13]:
machine_util = (
    pd.DataFrame.from_dict(machine_load, orient="index", columns=["Used Minutes"])
    .reset_index()
    .rename(columns={"index": "Machine"})
)

machine_util["Used Hours"] = machine_util["Used Minutes"] / 60
machine_util


,Machine,Used Minutes,Used Hours
0,M.P-09,1200.000000,20.000000
1,M.P-16,1165.714286,19.428571


In [14]:
df_reject = pd.DataFrame(rejection_log, columns=["Series", "Reason"])
df_reject["Count"] = 1

rejection_summary = (
    df_reject
    .groupby("Reason")
    .count()
    .sort_values("Count", ascending=False)
)

rejection_summary


,Series,Count
Reason,,
FG >= buffer,59747,59747
Machine not found,53871,53871
Machine capacity exceeded,728,728


In [15]:
top_2_reasons = rejection_summary.head(2)
top_2_reasons

,Series,Count
Reason,,
FG >= buffer,59747,59747
Machine not found,53871,53871


In [16]:
rejection_summary.head(5)


,Series,Count
Reason,,
FG >= buffer,59747,59747
Machine not found,53871,53871
Machine capacity exceeded,728,728
